In [1]:
# added top-p and top-k filtering in generate function
# set vocab_size in config.py
# MHA with KV cache + RoPE + PyTorch SDPA.
# This traditional implementation is easier to understand, and still efficient in practice.
# GQA and MLA is a great way for long-text inference with reduced KV cache size,
# but both comes with slight loss increase and no efficiency merits during training phase.
# KV cache does not help training speed. Codebase will be simpler without it.
# KV cache supports multi-turn continuation by RoPE with position offset.
# No Dropout. Dataset is large enough and regularization is not necessary.

import torch
import torch.nn as nn
import torch.nn.functional as F

class TokenEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.embedding_dim)
        # keep embedding in default dtype (autocast will handle bf16 when enabled)

    def forward(self, input_indices):
        return self.token_embedding_table(input_indices)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, rope_theta=1e6):
        super().__init__()

        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2) / dim))
        position_index = torch.arange(max_seq_len)
        frequency_matrix = torch.einsum('i,j->ij', position_index, inv_freq)

        cosine = torch.cos(frequency_matrix)[None, None, :, :]
        sine = torch.sin(frequency_matrix)[None, None, :, :]

        self.register_buffer("cos_cached", cosine, persistent=False)
        self.register_buffer("sin_cached", sine, persistent=False)

    def apply_rotary_emb(self, x, position_offset=0):
        sequence_length = x.size(2)

        cosine = self.cos_cached[:, :, position_offset:position_offset + sequence_length, :]
        sine = self.sin_cached[:, :, position_offset:position_offset + sequence_length, :]

        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        rotated_even = x_even * cosine - x_odd * sine
        rotated_odd = x_odd * cosine + x_even * sine

        rotated = torch.empty_like(x)
        rotated[..., 0::2] = rotated_even
        rotated[..., 1::2] = rotated_odd

        return rotated

class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_heads = config.num_attention_heads
        self.embed_dim = config.embedding_dim
        self.head_dim = self.embed_dim // self.num_heads

        # QKV projection
        self.query_fc = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
        self.key_fc   = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
        self.value_fc = nn.Linear(self.embed_dim, self.embed_dim, bias=False)

        # Rotary Positional Embedding (RoPE)
        self.rotary_emb = RotaryEmbedding(
            dim=self.head_dim,
            max_seq_len=config.max_sequence_length,
            rope_theta=config.rope_theta
        )

        self.output_projection = nn.Linear(self.embed_dim, self.embed_dim)

        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(
                config.max_sequence_length,
                config.max_sequence_length,
                dtype=torch.bool
            )),
            persistent=False
        )

        # KV cache
        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)
        self.current_pos = 0

    # --------------------------------------------------
    # router
    # --------------------------------------------------
    def forward(self, x, use_cache=False):
        input_len = x.size(1)
        if use_cache is False:
            return self.forward_no_cache(x)
        elif use_cache is True and input_len > 1:
            return self.forward_prefill(x)
        elif use_cache is True and input_len == 1: # Hi scenario also starts with T==1
            return self.forward_cached_decoding(x)
        else:
            raise RuntimeError("Unexpected condition in MultiHeadAttention forward")

    # --------------------------------------------------
    # (1) no cache : training 
    # --------------------------------------------------
    def forward_no_cache(self, x):
        B, T, C = x.shape

        Q = self.query_fc(x)
        K = self.key_fc(x)
        V = self.value_fc(x)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # RoPE : offset = 0
        Q = self.rotary_emb.apply_rotary_emb(Q, position_offset=0)
        K = self.rotary_emb.apply_rotary_emb(K, position_offset=0)

        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=None,
            is_causal=True
        )

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.output_projection(out)
        return out

    # --------------------------------------------------
    # (2) prefill : initialize KV cache
    # --------------------------------------------------
    def forward_prefill(self, x):
        B, T, C = x.shape

        Q = self.query_fc(x)
        K = self.key_fc(x)
        V = self.value_fc(x)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # init cache
        if self.cache_k is None:
            self.cache_k = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=K.dtype
            )
            self.cache_v = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=V.dtype
            )
            self.current_pos = 0

        # RoPE : offset = current_pos (supports multi-turn continuation)
        Q = self.rotary_emb.apply_rotary_emb(Q, position_offset=self.current_pos)
        K = self.rotary_emb.apply_rotary_emb(K, position_offset=self.current_pos)

        # prevent overflow
        if self.current_pos + T > self.config.max_sequence_length:
            raise RuntimeError("KV cache exceeded max_sequence_length")

        self.cache_k[:, :, self.current_pos:self.current_pos + T, :] = K
        self.cache_v[:, :, self.current_pos:self.current_pos + T, :] = V

        K = self.cache_k[:, :, :self.current_pos + T, :]
        V = self.cache_v[:, :, :self.current_pos + T, :]

        attn_mask = self.causal_mask[
            self.current_pos : self.current_pos + T,
            : self.current_pos + T
        ]

        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attn_mask,
            is_causal=False
        )

        self.current_pos += T

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.output_projection(out)
        return out

    # --------------------------------------------------
    # (3) decode : cached decoding (1 token)
    # --------------------------------------------------
    def forward_cached_decoding(self, x):
        B, T, C = x.shape
        assert T == 1, "cached decoding expects T==1"

        Q = self.query_fc(x)
        K = self.key_fc(x)
        V = self.value_fc(x)

        Q = Q.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)

        # This is not usually needed since prefill should have initialized the cache.
        # Just in case for "Hi" scenario, which starts with single token input.
        if self.cache_k is None:
            self.cache_k = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=K.dtype
            )
            self.cache_v = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=V.dtype
            )
            self.current_pos = 0

        if self.current_pos + 1 >= self.config.max_sequence_length:
            raise RuntimeError("KV cache exceeded max_sequence_length")

        # RoPE : offset = current_pos
        Q = self.rotary_emb.apply_rotary_emb(Q, position_offset=self.current_pos)
        K = self.rotary_emb.apply_rotary_emb(K, position_offset=self.current_pos)

        self.cache_k[:, :, self.current_pos:self.current_pos + 1, :] = K
        self.cache_v[:, :, self.current_pos:self.current_pos + 1, :] = V

        K = self.cache_k[:, :, :self.current_pos + 1, :]
        V = self.cache_v[:, :, :self.current_pos + 1, :]

        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=None,
            is_causal=False
        )

        self.current_pos += 1

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.output_projection(out)
        return out

    def reset_cache(self):
        self.cache_k = None
        self.cache_v = None
        self.current_pos = 0



class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()    
        self.net = nn.Sequential(
            nn.Linear(config.embedding_dim, config.hidden_dim, bias=False),
            nn.ReLU(),
            nn.Linear(config.hidden_dim, config.embedding_dim, bias=False),
        )

    def forward(self, input_tensor):
        return self.net(input_tensor)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(config.embedding_dim)
        self.layer_norm2 = nn.LayerNorm(config.embedding_dim)
        self.multihead_attention = MultiHeadAttention(config=config)
        self.feed_forward = FeedForward(config=config)


    def forward(self, input_tensor, use_cache=False):
        normed_input = self.layer_norm1(input_tensor)
        attention_output = self.multihead_attention(normed_input, use_cache=use_cache)
        residual_attention = attention_output + input_tensor
        normed_attention = self.layer_norm2(residual_attention)
        feedforward_output = self.feed_forward(normed_attention)
        final_output = feedforward_output + residual_attention
        return final_output


class VocabularyLogits(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.output_norm = nn.LayerNorm(config.embedding_dim)
        self.vocab_projection = nn.Linear(config.embedding_dim, config.vocab_size, bias=False)

    def forward(self, transformer_block_output):
        x = transformer_block_output
        normalized_output = self.output_norm(x)
        vocab_logits = self.vocab_projection(normalized_output)
        return vocab_logits


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding_layer = TokenEmbedding(config=config)
        self.blocks = nn.ModuleList([TransformerBlock(config=config) for _ in range(config.layer_count)])
        self.vocab_projection = VocabularyLogits(config=config)
        self.criterion = nn.CrossEntropyLoss()


    def forward(self, input_indices, target_indices, use_cache=False):
        token_embeddings = self.token_embedding_layer.forward(input_indices)

        x = token_embeddings
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        logits = self.vocab_projection(x)

        if target_indices is None:
            return logits, None

        batch_size, token_len, vocab_size = logits.shape
        logits_flat = logits.view(batch_size * token_len, vocab_size)
        targets_flat = target_indices.view(batch_size * token_len)
        loss = self.criterion(logits_flat, targets_flat)
        return logits, loss


    def generate(self,
        input_indices,
        max_new_tokens,
        temperature=1.0,
        use_cache=True,
        reset_cache=False,
        top_k=None,      # ### NEW ###
        top_p=None,      # ### NEW ###
    ):
        self.eval()

        if reset_cache:
            for block in self.blocks:
                block.multihead_attention.reset_cache()

        next_token = None

        for i in range(max_new_tokens):
            if use_cache:
                if i == 0:
                    logits, _ = self.forward(input_indices, None, use_cache=True)
                else:
                    logits, _ = self.forward(next_token, None, use_cache=True)
            else:
                logits, _ = self.forward(input_indices, None, use_cache=False)

            """ DELETE
            last_logits = logits[:, -1, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            """

            ### NEW ###
            last_logits = logits[:, -1, :] / temperature

            if top_k is not None:
                top_k = min(top_k, last_logits.size(-1))
                values, _ = torch.topk(last_logits, top_k)
                min_value = values[:, -1].unsqueeze(-1)
                last_logits = torch.where(
                    last_logits < min_value,
                    torch.full_like(last_logits, float("-inf")),
                    last_logits,
                )

            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(last_logits, descending=True)
                sorted_probs = F.softmax(sorted_logits, dim=-1)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                sorted_mask = cumulative_probs > top_p
                sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
                sorted_mask[..., 0] = False

                sorted_logits = torch.where(
                    sorted_mask,
                    torch.full_like(sorted_logits, float("-inf")),
                    sorted_logits,
                )

                last_logits = torch.zeros_like(last_logits).scatter(
                    -1, sorted_indices, sorted_logits
                )

            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            ### NEW ###

            yield int(next_token.item())
            input_indices = torch.cat((input_indices, next_token), dim=1)

In [2]:
class Config:
    embedding_dim: int = 2560
    hidden_dim: int = 10240
    num_attention_heads: int = 20
    layer_count: int = 30
    rope_theta: float = 1_000_000.0
    vocab_size: int = 50257
    max_sequence_length: int = 2048

In [3]:
config = Config()
model = GPT(config)

In [4]:
device = torch.device("cuda")
torch.set_float32_matmul_precision("high")

In [5]:
import random
RANDOM_SEED = 1337
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

In [6]:
from huggingface_hub import hf_hub_download, list_repo_files

REPO_ID = "HayatoHongo/AIkenSGTv1"
MODEL_FILENAME = "prompt_mask_instruction_tuned_model_epoch_2_lr_1e-04_gkentei_text.safetensors"
model_path = hf_hub_download(repo_id=REPO_ID, filename=MODEL_FILENAME)
print("Model:", MODEL_FILENAME)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


prompt_mask_instruction_tuned_model_epoc(…): reconstructing file:   0%|          |  0.00B / 10.5GB            

prompt_mask_instruction_tuned_model_epoc(…): downloading bytes:           |  0.00B            

Model: prompt_mask_instruction_tuned_model_epoch_2_lr_1e-04_gkentei_text.safetensors


In [7]:
import tiktoken
from safetensors.torch import load_file

tokenizer = tiktoken.get_encoding("gpt2")
state_dict = load_file(model_path, device="cpu")
model.load_state_dict(state_dict)

<All keys matched successfully>

In [17]:
model = model.to(device)
model.eval()

GPT(
  (token_embedding_layer): TokenEmbedding(
    (token_embedding_table): Embedding(50257, 2560)
  )
  (blocks): ModuleList(
    (0-29): 30 x TransformerBlock(
      (layer_norm1): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
      (layer_norm2): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
      (multihead_attention): MultiHeadAttention(
        (query_fc): Linear(in_features=2560, out_features=2560, bias=False)
        (key_fc): Linear(in_features=2560, out_features=2560, bias=False)
        (value_fc): Linear(in_features=2560, out_features=2560, bias=False)
        (rotary_emb): RotaryEmbedding()
        (output_projection): Linear(in_features=2560, out_features=2560, bias=True)
      )
      (feed_forward): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=2560, out_features=10240, bias=False)
          (1): ReLU()
          (2): Linear(in_features=10240, out_features=2560, bias=False)
        )
      )
    )
  )
  (vocab_projection): 

## test セット評価

In [ ]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="HayatoHongo/AIkenSGTv1",
    repo_type="model",
    filename="gkentei_eval1_prompt_response.jsonl",
    local_dir=".",
)

(…)v2_direct_fewshot_full_sorted_test.jsonl:   0%|          | 0.00/69.4k [00:00<?, ?B/s]

'/content/mock_questions_v2_direct_fewshot_full_sorted_test.jsonl'

In [ ]:
from google.colab import files
from pathlib import Path
import json

TEST_PATH = Path("/content/gkentei_eval1_prompt_response.jsonl")
with TEST_PATH.open(encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]


In [ ]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="HayatoHongo/AIkenSGTv1",
    repo_type="model",
    filename="/content/gkentei_eval1_prompt_response.jsonl",
    local_dir=".",
)

(…)_v2_direct_fewshot_full_sorted_dev.jsonl:   0%|          | 0.00/2.81k [00:00<?, ?B/s]

'/content/mock_questions_v2_direct_fewshot_full_sorted_dev.jsonl'

In [14]:
DEV_PATH = Path("/content/mock_questions_v2_direct_fewshot_full_sorted_dev.jsonl")
with DEV_PATH.open(encoding="utf-8") as f:
    dev_data = [json.loads(line) for line in f]

FEW_SHOT_COUNT = 0  # 0〜5
few_shot_prompt = "".join(
    f"<USER>{item['prompt']}<ASSISTANT>{item['response']}<|endoftext|>"
    for item in dev_data[:FEW_SHOT_COUNT]
)


In [18]:
import pandas as pd
from IPython.display import display

results = []
for i, item in enumerate(test_data, start=1):
    prompt = few_shot_prompt + "<USER>" + item["prompt"] + "<ASSISTANT>"
    input_ids = torch.tensor([tokenizer.encode(prompt, allowed_special="all")], device=device)
    output_ids = []

    with torch.inference_mode():
        for token in model.generate(input_ids, max_new_tokens=512, top_k=None, temperature = 0.1, reset_cache=True):
            if token == tokenizer.eot_token:
                break
            output_ids.append(token)

    raw_output = tokenizer.decode(output_ids)
    results.append({
        "row": i,
        "prompt": item["prompt"],
        "expected_response": item["response"],
        "raw_output": raw_output,
        "exact_match": raw_output == item["response"],
    })

results = pd.DataFrame(results)
OUTPUT_PATH = "/content/gkentei_test_results_eval1.csv"
results.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"完全一致: {results['exact_match'].sum()}/{len(results)}")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
display(results)

完全一致: 35/109


,row,prompt,expected_response,raw_output,exact_match
0,1,AIシステムのモニタリングが適切に行われなかった場合の影響について述べたものとして、最も適切な選択肢を1つ選べ\nA. モデルの開発コストが大幅に削減される\nB. モデルの精度が自動的に向上する\nC. モデルの学習時間が短縮される\nD. モデルの劣化やバイアスの発生を検知できず、AIの信頼性が損なわれる,\boxed{モデルの劣化やバイアスの発生を検知できず、AIの信頼性が損なわれる},\boxed{モデルの学習時間が短縮される},False
1,2,多腕バンディット問題において、探索と活用のバランスを取るために、各腕の期待報酬の信頼区間の上限に基づいて腕を選択する方策として、最も適切な選択肢を1つ選べ\nA. ε-greedy 方策\nB. トンプソンサンプリング\nC. UCB 方策\nD. ランダム方策,\boxed{UCB 方策},\boxed{UCB 方策},True
2,3,勾配ブースティングにおいて、過学習が起こりやすくなる条件として、最も適切な選択肢を1つ選べ\nA. 学習率（shrinkage）を小さく設定する\nB. 弱学習器の数を少なくする\nC. 木の深さを大きくし、学習率を高く設定する\nD. 正則化パラメータを強くする,\boxed{木の深さを大きくし、学習率を高く設定する},\boxed{弱学習器の数を少なくする},False
3,4,以下の文章を読み、空欄に最もよく当てはまる選択肢を1つ選べ\n線形回帰モデルの学習では、一般に（ ）と呼ばれる、予測値と実測値の誤差の二乗和を最小化する手法が用いられる。\nA. 最尤推定法\nB. 最小二乗法\nC. 勾配降下法\nD. 正則化法,\boxed{最小二乗法},\boxed{最小二乗法},True
4,5,準委任契約に関する説明として、最も適切な選択肢を1つ選べ\nA. 仕事の完成を約束し、成果物の引渡しに対して報酬が支払われる契約である\nB. 法律行為の委託を対象とする契約である\nC. 受託者は善良な管理者の注意義務を負わない\nD. 法律行為でない事務の処理を委託する契約であり、受託者は善管注意義務を負うが、成果物の完成を保証しない,\boxed{法律行為でない事務の処理を委託する契約であり、受託者は善管注意義務を負うが、成果物の完成を保証しない},\boxed{法律行為でない事務の処理を委託する契約であり、受託者は善管注意義務を負うが、成果物の完成を保証しない},True
5,6,音声合成システムの運用中に、入力テキストの傾向が変化したことで合成音声の自然性が低下した。この現象の原因として最も適切な選択肢を1つ選べ\nA. 音声合成モデルの学習データが多すぎた\nB. テキスト解析部の精度が向上した\nC. データドリフトが発生し、モデルの性能が劣化した\nD. ボコーダーの処理速度が速すぎた,\boxed{データドリフトが発生し、モデルの性能が劣化した},\boxed{ボコーダーの処理速度が速すぎた},False
6,7,2つの変数間に統計的な相関が認められるが、実際には直接の因果関係がなく、第三の変数によって両者が影響を受けているために生じる見かけ上の相関を指す用語として、最も適切な選択肢を1つ選べ\nA. 因果関係\nB. 疑似相関\nC. 偏相関\nD. 交絡,\boxed{疑似相関},\boxed{疑似相関},True
7,8,ユークリッド距離を用いたk近傍法において、次元が高くなると性能が低下しやすくなる主な原因として、最も適切な選択肢を1つ選べ\nA. 次元の呪いにより、点間の距離の差が小さくなるため\nB. 計算量が次元に比例して増加するため\nC. ユークリッド距離が高次元では定義できないため\nD. マンハッタン距離の方が常に優れているため,\boxed{次元の呪いにより、点間の距離の差が小さくなるため},\boxed{ユークリッド距離が高次元では定義できないため},False
8,9,以下の文章を読み、空欄に最もよく当てはまる選択肢を1つ選べ\nQuestion-Answeringシステムでは、ユーザーの質問に対して適切な回答を返すために、しばしば（ ）と呼ばれる、質問の意図を解析し、知識ベースや文書集合から回答を抽出・生成する処理パイプラインが用いられる。\nA. 情報検索\nB. チャットボット\nC. キーワードマッチング\nD. 質問応答,\boxed{質問応答},\boxed{質問応答},True
9,10,DQN（Deep Q-Network）に関する説明として、最も適切な選択肢を1つ選べ\nA. Q学習のテーブルを深層ニューラルネットワークで置き換えただけの手法である\nB. 連続行動空間の問題にそのまま適用できる手法である\nC. エージェントの経験を順番に学習することで、データの相関を保つ手法である\nD. 経験再生とターゲットネットワークにより学習の安定性を向上させる手法である,\boxed{経験再生とターゲットネットワークにより学習の安定性を向上させる手法である},\boxed{連続行動空間の問題にそのまま適用できる手法である},False


In [ ]:
files.download(OUTPUT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from collections import Counter

BATCH_SIZE = 16
N_VOTES = 32
SC_OUTPUT_PATH = "/content/gkentei_test_results_eval1temp03_sc32_b16.csv"
torch.manual_seed(1337)
torch.cuda.manual_seed_all(1337)

sc_results = []
for i, item in enumerate(test_data, start=1):
    prompt = few_shot_prompt + "<USER>" + item["prompt"] + "<ASSISTANT>"
    prompt_ids = tokenizer.encode(prompt, allowed_special="all")
    samples = []

    for start in range(0, N_VOTES, BATCH_SIZE):
        batch_size = min(BATCH_SIZE, N_VOTES - start)
        input_ids = torch.tensor([prompt_ids] * batch_size, dtype=torch.long, device=device)
        for block in model.blocks:
            block.multihead_attention.reset_cache()

        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
        generated = []
        next_token = input_ids
        with torch.inference_mode():
            for _ in range(512):
                logits, _ = model(next_token, None, use_cache=True)
                probs = torch.softmax(logits[:, -1, :] / 0.3, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                next_token = torch.where(finished[:, None], tokenizer.eot_token, next_token)
                generated.append(next_token)
                finished |= next_token[:, 0] == tokenizer.eot_token
                if finished.all():
                    break

        for ids in torch.cat(generated, dim=1).tolist():
            if tokenizer.eot_token in ids:
                ids = ids[:ids.index(tokenizer.eot_token)]
            samples.append(tokenizer.decode(ids))

    votes = Counter(samples).most_common()
    tie = len(votes) > 1 and votes[0][1] == votes[1][1]
    majority = "" if tie else votes[0][0]
    sc_results.append({
        "row": i,
        "prompt": item["prompt"],
        "expected_response": item["response"],
        "raw_samples": json.dumps(samples, ensure_ascii=False),
        "majority_output": majority,
        "vote_count": votes[0][1],
        "tie": tie,
        "exact_match": majority == item["response"],
    })
    if i % 10 == 0:
        print(f"{i}/{len(test_data)}")

sc_results = pd.DataFrame(sc_results)
sc_results.to_csv(SC_OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"多数決の完全一致: {sc_results['exact_match'].sum()}/{len(sc_results)}")
display(sc_results)
files.download(SC_OUTPUT_PATH)